# FDS 부정사용 탐지 — FT-Transformer (팀 제출본 v2)

산학협력 프로젝트 · 딥러닝 파트

## v2 변경점

**`DELNG_MONTH` · `DELNG_QUARTER` 를 피처에서 제외**했습니다. 두 컬럼은 라벨 성숙도 누수입니다 — 최근 결제일수록 조사가 덜 끝나 양성률이 낮게 찍히므로, 모델이 "부정 여부"가 아니라 "조사 진행도"를 학습하게 됩니다. 운영 시점에는 재현되지 않는 신호라 팀 공통으로 제외합니다.

성능 지표가 v1보다 **내려가는 것이 정상**입니다. 내려간 만큼이 누수분이었다는 뜻입니다.

## 평가 프로토콜

| 단계 | 사용 데이터 |
|---|---|
| 전처리 기준값 (중앙값·분위수·카테고리 사전) | **train 만** |
| early stopping | **valid-tune** (valid의 30%) |
| 배포용 threshold | **valid-tune** |
| 성능 보고 | **valid-report** (valid의 70%, 교차적합 threshold) |
| test | **최종 채점 1회만** — 어떤 선택에도 미사용 |

train 에서 valid·test 와 겹치는 행은 로드 단계에서 자동 제거됩니다(`row_id` 기준). 제거 건수를 반드시 확인하세요.

## 보고 구간

팀 공지 기준에 따라 **전체 valid** 와 **신규 이용자 구간**을 나눠 보고합니다. 신규 = valid 의 `TRGTER_ID` 가 train 의 `TRGTER_ID` 집합에 없는 행. 두 구간의 격차가 크면 사람 단위 암기에 의존하고 있다는 신호입니다.

## 제출 파일 (`OUT_DIR`)

| 파일 | 내용 |
|---|---|
| `ft_transformer.pt` | 모델 가중치 · 피처 목록 · threshold |
| `preprocess.pkl` | cat_maps · 중앙값 · QuantileTransformer |
| `dl_predict.py` | `predict_dl(df)` → 0~1 점수 |
| `dl_scores_valid.csv` | `row_id, y, score` |
| `dl_scores_test.csv` | `row_id, y, score` |

**`.pt` · `.pkl` · `.py` 는 세 개가 한 세트입니다.** 전처리 객체 없이는 모델을 못 쓰고, 모델 정의 없이는 가중치를 못 읽습니다.

---

> **실행 전**: 런타임 → 런타임 유형 변경 → **GPU (T4)** 선택

In [ ]:
# ============================================================
# 0. Config — 경로/설정은 여기서만 수정
# ============================================================
# Drive 사용 시:
# from google.colab import drive
# drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/fds"          # ← 본인 폴더로 수정

TRAIN_PATH = f"{BASE}/train.parquet"         # 학습 파일 (없으면 5.5차 파일 경로를 넣으세요)
VALID_PATH = f"{BASE}/valid.parquet"         # 검증 파일
TEST_PATH  = f"{BASE}/test.parquet"          # 테스트 파일 — 용량이 커도 됩니다(청크 처리)

OUT_DIR    = f"{BASE}/outputs"               # 결과 저장 위치

TARGET   = "INJSTUSE_AT"                     # 부정사용 여부 (0/1)
ID_COLS  = ["row_id", "DELNG_SN", "CONFM_NO"]  # 식별자 — 피처 제외, 결과에는 포함
LEAK_KEY = "CONFM_NO"                        # 누수 제거 기준 (row_id 가 양쪽에 있으면 자동으로 그걸 사용)

# --- ⚠️ 피처에서 반드시 제외할 컬럼 ---
#  · 사람 ID 를 피처로 쓰면 "이 대상자는 부정사용자"를 통째로 암기한다.
#    검증셋 실측에서 이 효과만으로 F1 이 71% 부풀려졌다(0.33 → 0.56).
#  · 원본 타임스탬프는 거의 모든 행이 고유(99%)해서 학습 신호가 없고 암기만 유발한다.
#    시간 정보는 F_hour_sin/cos·DELNG_MONTH 등 파생변수가 이미 담고 있다.
#  · 달력 컬럼(DELNG_MONTH/QUARTER)은 라벨 성숙도 컨닝이다. 최근 결제일수록 조사가
#    덜 끝나 양성률이 낮게 찍히므로, 모델이 "부정 여부"가 아니라 "조사 진행도"를
#    학습한다. 운영 시점에는 재현되지 않는 신호라 반드시 뺀다. (팀 공통 제외 컬럼)
EXCLUDE_COLS = [
    "TRGTER_ID", "PRHNF_ID",             # 대상자·제공인력 식별자 (그룹 CV 용도로만 사용)
    "DELNG_DT",                          # 거래 일시 (문자열, 고유 99.9%)
    "SVC_BEGIN_DT", "SVC_END_DT",        # 서비스 시작·종료 시각 (정수 타임스탬프, 고유 99%)
    "DELNG_DE", "PREDELNG_DE",           # 거래·직전거래 일자
    "DELNG_MONTH", "DELNG_QUARTER",      # ⚠️ 라벨 성숙도 누수 — 팀 공통 제외
]
# 범주형 자동 가드: 고유값이 전체 행의 이 비율을 넘으면 식별자로 보고 제외
CAT_UNIQUE_LIMIT = 0.20

CFG = {
    # --- 모델 ---
    "d_model": 96,
    "n_layers": 3,
    "n_heads": 8,
    "ffn_mult": 4/3,
    "dropout": 0.15,

    # --- 학습 ---
    "epochs": 40,
    "batch_size": 1024,
    "lr": 1e-4,
    "weight_decay": 1e-5,
    "warmup_ratio": 0.05,
    "patience": 6,
    "grad_clip": 1.0,
    "seed": 42,
    "num_workers": 0,          # 0 이 안전. 워커 프로세스가 메모리를 추가로 잡아 런타임이 끊길 수 있음

    # --- 메모리 절약 ---
    "qt_subsample": 200_000,   # QuantileTransformer fit 표본 수 (전체를 쓰면 메모리 급증)
    "free_dataframes": True,   # 전처리 후 원본 DataFrame 해제 (RAM 1~2GB 절약)

    # --- 체크포인트 (런타임이 끊겨도 이어서 학습) ---
    #  Drive 에 매 epoch 쓰면 FUSE 마운트가 멈추는 일이 잦다.
    #  로컬(/content)에 매 epoch 저장하고, Drive 에는 가끔만 복사한다.
    "resume": True,
    "ckpt_drive_every": 5,     # 몇 epoch 마다 Drive 로 복사할지 (0 이면 학습 끝날 때만)

    # --- 검증 분할 ---
    "val_tune_ratio": 0.3,     # valid 중 early stopping·컷오프 산출에 쓸 비율
                               # 나머지 70% 는 성능 보고 전용

    # --- 테스트 추론 ---
    "infer_chunk_rows": 200_000,   # 한 번에 읽을 행 수 (메모리 부족하면 줄이세요)
}

In [ ]:
# ============================================================
# 1. Imports & Seed
# ============================================================
import os, math, random, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score, roc_auc_score,
    confusion_matrix, classification_report,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU 런타임이 아닙니다. 런타임 → 런타임 유형 변경 → GPU 선택 권장")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
BASE = "/content/drive/MyDrive/fds"
print("Drive 마운트:", os.path.isdir("/content/drive/MyDrive"))
print(f"\n{BASE} 안의 parquet:")
for f in sorted(glob.glob(f"{BASE}/*.parquet")):
    print(f"   {os.path.basename(f):45s} {os.path.getsize(f)/1024**2:8.1f} MB")

TRAIN_PATH = f"{BASE}/train.parquet"
VALID_PATH = f"{BASE}/valid.parquet"
TEST_PATH  = f"{BASE}/test.parquet"
OUT_DIR    = f"{BASE}/outputs"

# ============================================================
# 1-b. 실행 전 환경 점검 — 문제가 있으면 여기서 멈춥니다
# ============================================================
import importlib

problems = []

try:
    importlib.import_module("pyarrow")
    print("✅ pyarrow 사용 가능")
except ImportError:
    problems.append("pyarrow 없음 →  !pip install pyarrow  후 [런타임 → 세션 다시 시작]")
    print("❌ pyarrow 없음")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")
else:
    problems.append("GPU 미할당 → [런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU]")
    print("❌ GPU 미할당 (CPU 는 수 시간 소요)")
print(f"   torch {torch.__version__} / pandas {pd.__version__} / numpy {np.__version__}")

need = [("TRAIN_PATH", TRAIN_PATH, True), ("VALID_PATH", VALID_PATH, True),
        ("TEST_PATH",  TEST_PATH,  False)]   # test 는 없어도 학습·평가는 진행 가능
missing_required = False
for label, path, required in need:
    if os.path.exists(path):
        print(f"✅ {label}: {os.path.basename(path)}  ({os.path.getsize(path)/1024**2:.1f} MB)")
    elif required:
        print(f"❌ {label} 없음: {path}")
        missing_required = True
    else:
        print(f"⚠️ {label} 없음: {path}  (테스트 추론 셀은 건너뜁니다)")

if missing_required:
    found, roots = [], ["/content"]
    if os.path.isdir("/content/drive/MyDrive"):
        roots.append("/content/drive/MyDrive")
    else:
        print("   ℹ️ Drive 미마운트 "
              "(from google.colab import drive; drive.mount('/content/drive'))")
    for root in roots:
        b = root.rstrip("/").count("/")
        for dp, dn, fn in os.walk(root):
            if dp.rstrip("/").count("/") - b >= 3:
                dn[:] = []; continue
            dn[:] = [d for d in dn if not d.startswith(".")
                     and d not in ("sample_data", "outputs", "__pycache__", "drive")]
            found += [os.path.join(dp, f) for f in fn if f.endswith(".parquet")]
            if len(found) >= 30: break
    if found:
        print("\n   🔍 찾은 parquet:")
        for f in found[:30]:
            print(f"      {f}   ({os.path.getsize(f)/1024**2:.1f} MB)")
        print("\n   ▶ 위 경로를 Config 셀의 BASE / *_PATH 에 반영하세요.")
    else:
        print("\n   🔍 parquet 을 찾지 못했습니다. 업로드하거나 Drive 를 마운트하세요.")
    problems.append("데이터 파일 경로 불일치 → Config 셀 수정")

if problems:
    msg = "\n".join(f"  {i}. {m}" for i, m in enumerate(problems, 1))
    raise RuntimeError(f"\n\n▶ 실행을 멈췄습니다. 아래 {len(problems)}건을 해결하세요:\n{msg}\n"
                       "\n해결 후 [런타임 → 모두 실행] 으로 다시 실행하세요.\n")
print("\n▶ 준비 완료.")

In [ ]:
# ============================================================
# 2. 데이터 로드 · 누수 제거 · 검증셋 분할
# ============================================================
train_df = pd.read_parquet(TRAIN_PATH)
valid_df = pd.read_parquet(VALID_PATH)
print(f"train 원본: {train_df.shape[0]:,}행 × {train_df.shape[1]}컬럼  "
      f"(양성률 {train_df[TARGET].mean():.2%})")
print(f"valid     : {valid_df.shape[0]:,}행 × {valid_df.shape[1]}컬럼  "
      f"(양성률 {valid_df[TARGET].mean():.2%})")

# --- 2-1. ⚠️ 누수 제거 ---
# valid·test 의 행이 train 안에 들어 있으면 정답을 미리 보고 학습하게 된다.
# row_id 가 양쪽에 있으면 그 기준으로, 없으면 승인번호로 제거한다.
def _keys(df, key):
    return set(df[key].tolist()) if key in df.columns else set()

leak_key = "row_id" if ("row_id" in train_df.columns and "row_id" in valid_df.columns) else LEAK_KEY
holdout_keys = _keys(valid_df, leak_key)
if os.path.exists(TEST_PATH):
    try:
        import pyarrow.parquet as pq
        _tk = pq.read_table(TEST_PATH, columns=[leak_key])[leak_key].to_pylist()
        holdout_keys |= set(_tk)
        print(f"   test 의 {leak_key} {len(set(_tk)):,}개도 제외 대상에 포함")
    except Exception as e:
        print(f"   ⚠️ test 키를 읽지 못했습니다({type(e).__name__}). valid 기준으로만 제거합니다.")

before = len(train_df)
if leak_key in train_df.columns and holdout_keys:
    train_df = train_df[~train_df[leak_key].isin(holdout_keys)].reset_index(drop=True)
removed = before - len(train_df)
print(f"\n[누수 제거] 기준={leak_key}  →  {removed:,}행 제거 ({removed/max(before,1):.1%})")
print(f"   train: {before:,} → {len(train_df):,}행  (양성률 {train_df[TARGET].mean():.2%})")
if removed == 0:
    print("   ℹ️ 제거된 행이 없습니다. 이미 분리된 train 파일을 쓰고 계신 것으로 보입니다.")
elif removed / before > 0.5:
    print("   ⚠️ 절반 이상이 제거되었습니다. train 파일이 올바른지 확인하세요.")

# --- 2-2. 피처 결정 (train ∩ valid, 타깃·식별자·원본일시 제외) ---
_drop = set(ID_COLS) | set(EXCLUDE_COLS) | {TARGET}
FEATURES = [c for c in train_df.columns if c in set(valid_df.columns) and c not in _drop]

only_tr = sorted(set(train_df.columns) - set(valid_df.columns) - _drop)
only_va = sorted(set(valid_df.columns) - set(train_df.columns) - _drop)
excluded_present = [c for c in EXCLUDE_COLS if c in train_df.columns and c in valid_df.columns]

print(f"\n사용 피처 {len(FEATURES)}개")
if excluded_present:
    print(f"   식별자·원본일시로 제외 ({len(excluded_present)}): {excluded_present}")
if only_tr: print(f"   train 에만 있어 제외 ({len(only_tr)}): {only_tr[:8]}")
if only_va: print(f"   valid 에만 있어 제외 ({len(only_va)}): {only_va[:8]}")

# --- 2-3. 정렬 상태 확인 ---
for name, d in [("train", train_df), ("valid", valid_df)]:
    if d[TARGET].is_monotonic_increasing:
        print(f"⚠️ {name} 은 타깃 기준 정렬 상태 — 층화 셔플 분할 필수 (아래에서 처리)")

# --- 2-4. valid 를 tune / report 로 분할 ---
#   tune   → early stopping, threshold·등급 컷오프 산출
#   report → 최종 성능 보고 (선택 과정에 일절 미사용)
y_valid = valid_df[TARGET].astype(np.float32).values
idx_vt, idx_vr = train_test_split(
    np.arange(len(valid_df)), test_size=1 - CFG["val_tune_ratio"],
    stratify=y_valid, random_state=CFG["seed"])
print(f"\nvalid 분할 — tune {len(idx_vt):,}행(양성 {int(y_valid[idx_vt].sum()):,}) / "
      f"report {len(idx_vr):,}행(양성 {int(y_valid[idx_vr].sum()):,})")

# --- 2-5. 사람 단위 그룹 정보 (누수 진단용) ---
GROUP_COL = next((c for c in ("TRGTER_ID", "PRHNF_ID") if c in valid_df.columns), None)
if GROUP_COL:
    _g = valid_df[GROUP_COL]
    print(f"\n{GROUP_COL} 보유 — 고유 {_g.nunique():,}명, "
          f"2건 이상 {(_g.value_counts() > 1).mean():.1%}")
    print("   ※ valid 로 교차검증할 때는 StratifiedGroupKFold(groups=" + GROUP_COL + ") 를 쓰세요.")
    print("     무작위 분할은 같은 사람 암기로 성능이 크게 부풀려집니다.")

# --- 2-6. 신규 이용자 구간 (팀 공지 기준 보고용) ---
#   train 의 TRGTER_ID 집합에 없는 valid 행 = "train 에서 본 적 없는 이용자"
#   ⚠️ train_df 는 전처리 셀에서 메모리 해제되므로 여기서 미리 집합을 잡아둔다.
SEG_COL = "TRGTER_ID"
if SEG_COL in train_df.columns and SEG_COL in valid_df.columns:
    train_trgter = set(train_df[SEG_COL].unique().tolist())
    is_new_user  = ~valid_df[SEG_COL].isin(train_trgter).values     # valid 행 단위 bool
    _n_new = int(is_new_user.sum())
    print(f"\n[구간 분리] train 고유 {SEG_COL} {len(train_trgter):,}명 기준")
    if _n_new == 0:
        print("   ⚠️ 신규 이용자 0행 — valid 의 모든 이용자가 train 에도 있습니다.")
    elif _n_new == len(valid_df):
        print("   ℹ️ valid 전원이 신규 이용자입니다 (사람 단위로 분할된 데이터).")
    else:
        print(f"   신규 이용자 {_n_new:,}행 ({_n_new/len(valid_df):5.1%}) — "
              f"양성률 {y_valid[is_new_user].mean():.2%}")
        print(f"   기존 이용자 {len(valid_df)-_n_new:,}행 ({1-_n_new/len(valid_df):5.1%}) — "
              f"양성률 {y_valid[~is_new_user].mean():.2%}")
else:
    train_trgter, is_new_user = set(), np.zeros(len(valid_df), dtype=bool)
    print(f"\n⚠️ {SEG_COL} 이 없어 신규/기존 구간을 나눌 수 없습니다 — 전체 수치만 보고됩니다.")

In [ ]:
# ============================================================
# 3. 전처리 — 모든 기준값은 train 에서만 산출
# ============================================================
def is_categorical(s: pd.Series) -> bool:
    return (not pd.api.types.is_numeric_dtype(s)) or pd.api.types.is_bool_dtype(s)

cat_cols = [c for c in FEATURES if is_categorical(train_df[c])]
num_cols = [c for c in FEATURES if c not in cat_cols]

# --- 3-0. 식별자성 컬럼 자동 가드 ---
# EXCLUDE_COLS 에 미처 적지 못한 ID·타임스탬프성 컬럼을 잡아낸다.
# 범주형인데 고유값이 지나치게 많으면 학습 신호가 아니라 행 식별자에 가깝다.
id_like = []
for c in cat_cols:
    ratio = train_df[c].nunique(dropna=False) / len(train_df)
    if ratio > CAT_UNIQUE_LIMIT:
        id_like.append((c, ratio))
if id_like:
    print(f"⚠️ 식별자성 범주형 자동 제외 ({len(id_like)}개) — 고유값 비율이 "
          f"{CAT_UNIQUE_LIMIT:.0%} 초과")
    for c, r in sorted(id_like, key=lambda x: -x[1]):
        print(f"     {c:28s} 고유값 비율 {r:6.1%}")
    cat_cols = [c for c in cat_cols if c not in {c_ for c_, _ in id_like}]

# 수치형은 연속값도 고유값이 많을 수 있어 자동 제외하지 않고 경고만 한다.
num_warn = [(c, train_df[c].nunique(dropna=False) / len(train_df)) for c in num_cols
            if train_df[c].nunique(dropna=False) / len(train_df) > 0.95]
if num_warn:
    print(f"\nℹ️ 고유값이 95% 넘는 수치형 {len(num_warn)}개 — 타임스탬프·일련번호라면 "
          f"EXCLUDE_COLS 에 추가하세요")
    for c, r in num_warn[:8]:
        print(f"     {c:28s} {r:6.1%}   예시 {train_df[c].iloc[:2].tolist()}")

# train 에서 상수인 컬럼 제거 (정보 없음)
const_train = [c for c in FEATURES if train_df[c].nunique(dropna=False) <= 1]
# valid 에서 상수인 컬럼도 제거 (전이 불가) — 라벨이 아닌 '값'만 보므로 평가 오염 아님
const_valid = [c for c in FEATURES
               if valid_df[c].nunique(dropna=False) <= 1 and train_df[c].nunique(dropna=False) > 1]
drop_cols = sorted(set(const_train) | set(const_valid))
cat_cols = [c for c in cat_cols if c not in drop_cols]
num_cols = [c for c in num_cols if c not in drop_cols]

if const_train: print(f"train 상수 제거 ({len(const_train)}): {const_train}")
if const_valid: print(f"valid 상수 제거 ({len(const_valid)}): {const_valid}")
print(f"→ 범주형 {len(cat_cols)}개, 수치형 {len(num_cols)}개 "
      f"(토큰 {1 + len(cat_cols) + len(num_cols)}개)")

# --- 3-1. 범주형 인코딩 (0 = 미등장 값 UNK) ---
def norm_cat(s: pd.Series) -> pd.Series:
    return (s.astype(str).str.strip().str.lower()
             .replace({"nan": "__na__", "none": "__na__"}))

cat_maps = {}
for c in cat_cols:
    cats = norm_cat(train_df[c]).unique().tolist()
    cat_maps[c] = {v: i + 1 for i, v in enumerate(sorted(cats))}

def encode_cat(d: pd.DataFrame) -> np.ndarray:
    # int32 로 저장해 메모리를 절반으로 (카테고리 수가 20억을 넘을 일은 없음)
    out = np.zeros((len(d), len(cat_cols)), dtype=np.int32)
    for j, c in enumerate(cat_cols):
        out[:, j] = norm_cat(d[c]).map(cat_maps[c]).fillna(0).astype(np.int32).values
    return out

cardinalities = [len(cat_maps[c]) + 1 for c in cat_cols]

# --- 3-2. 수치형 ---
num_medians = train_df[num_cols].median()

def prep_num_raw(d: pd.DataFrame) -> np.ndarray:
    x = d[num_cols].astype(np.float64)
    return x.replace([np.inf, -np.inf], np.nan).fillna(num_medians).values

qt = QuantileTransformer(n_quantiles=min(1000, len(train_df)),
                         output_distribution="normal",
                         subsample=CFG["qt_subsample"],   # 전체를 정렬하면 메모리가 급증
                         random_state=CFG["seed"])

X_num_train = qt.fit_transform(prep_num_raw(train_df)).astype(np.float32)
X_num_valid = qt.transform(prep_num_raw(valid_df)).astype(np.float32)
X_cat_train = encode_cat(train_df)
X_cat_valid = encode_cat(valid_df)
y_train = train_df[TARGET].astype(np.float32).values
y_valid = valid_df[TARGET].astype(np.float32).values

X_num_vt, X_cat_vt, y_vt = X_num_valid[idx_vt], X_cat_valid[idx_vt], y_valid[idx_vt]
X_num_vr, X_cat_vr, y_vr = X_num_valid[idx_vr], X_cat_valid[idx_vr], y_valid[idx_vr]

# --- 3-3. UNK 비율 진단 (전처리 불일치 조기 발견) ---
unk = (X_cat_valid == 0).mean(axis=0)
bad = [(cat_cols[j], unk[j]) for j in np.argsort(-unk) if unk[j] > 0.01]
if bad:
    print("\n[valid UNK 비율 > 1%]")
    for c, r in bad[:10]:
        print(f"   {c:32s} {r:6.2%}{'  ⚠️ 전처리 불일치 의심' if r > 0.2 else ''}")
else:
    print("\nvalid UNK 비율 이상 없음 ✅")

prior_train  = float(y_train.mean())
prior_target = float(y_valid.mean())
print(f"\nprior: train {prior_train:.4f} / valid {prior_target:.4f}  "
      f"({prior_train/max(prior_target,1e-9):.1f}배)")

# --- 3-4. 메모리 정리 ---
# 전처리가 끝나면 원본 train DataFrame 은 더 이상 필요 없다.
# 52만 행 × 100컬럼(문자열 포함)은 1~2GB 를 차지해 런타임 종료의 주원인이 된다.
import gc
def _mem():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1024**3
    except Exception:
        return float("nan")

before_mem = _mem()
if CFG["free_dataframes"]:
    del train_df
    gc.collect()
    print(f"\ntrain DataFrame 해제 — RAM {before_mem:.2f} → {_mem():.2f} GB")

for nm, a in [("X_num_train", X_num_train), ("X_cat_train", X_cat_train),
              ("X_num_valid", X_num_valid), ("X_cat_valid", X_cat_valid)]:
    print(f"   {nm:12s} {a.shape}  {a.dtype}  {a.nbytes/1024**2:7.1f} MB")
print(f"   현재 RAM 사용량 {_mem():.2f} GB")

In [ ]:
# ============================================================
# 4. Dataset / DataLoader
# ============================================================
def make_loader(X_num, X_cat, y, batch_size, shuffle):
    # 범주형은 int32 그대로 보관하고, 배치 단위로 GPU 에 올릴 때만 long 으로 변환한다.
    # (여기서 astype(int64) 를 하면 전체 복사본이 생겨 절약 효과가 사라진다)
    ds = TensorDataset(torch.from_numpy(X_num),
                       torch.from_numpy(X_cat),
                       torch.from_numpy(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=CFG["num_workers"],
                      pin_memory=(CFG["num_workers"] > 0), drop_last=False)

train_loader = make_loader(X_num_train, X_cat_train, y_train, CFG["batch_size"], shuffle=True)
vt_loader    = make_loader(X_num_vt, X_cat_vt, y_vt, CFG["batch_size"] * 2, shuffle=False)
vr_loader    = make_loader(X_num_vr, X_cat_vr, y_vr, CFG["batch_size"] * 2, shuffle=False)

pos_weight_value = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print(f"pos_weight = {pos_weight_value:.3f}  (train 양성률 {y_train.mean():.2%})")

In [ ]:
# ============================================================
# 5. FT-Transformer 모델
#    - 수치 피처: 피처별 (weight, bias)로 d_model 차원 토큰화
#    - 범주 피처: 피처별 Embedding
#    - [CLS] 토큰 + Transformer Encoder → CLS 출력으로 분류
#    참고: Gorishniy et al., "Revisiting Deep Learning Models for
#          Tabular Data" (NeurIPS 2021)
# ============================================================

class FeatureTokenizer(nn.Module):
    def __init__(self, n_num: int, cardinalities: list, d_model: int):
        super().__init__()
        self.n_num = n_num
        # 수치형: x_i * w_i + b_i  (피처별 선형 토큰화)
        self.num_weight = nn.Parameter(torch.empty(n_num, d_model))
        self.num_bias   = nn.Parameter(torch.empty(n_num, d_model))
        nn.init.normal_(self.num_weight, std=0.02)
        nn.init.normal_(self.num_bias,   std=0.02)
        # 범주형: 전체를 하나의 embedding table 로 (offset 방식 — 빠름)
        self.cardinalities = cardinalities
        if len(cardinalities) > 0:
            offsets = np.cumsum([0] + cardinalities[:-1])
            self.register_buffer("cat_offsets", torch.tensor(offsets, dtype=torch.long))
            self.cat_emb = nn.Embedding(sum(cardinalities), d_model)
            nn.init.normal_(self.cat_emb.weight, std=0.02)
        # [CLS]
        self.cls = nn.Parameter(torch.empty(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)

    def forward(self, x_num, x_cat):
        B = x_num.size(0)
        tokens = [self.cls.expand(B, -1, -1)]
        if self.n_num > 0:
            tokens.append(x_num.unsqueeze(-1) * self.num_weight + self.num_bias)  # (B, n_num, d)
        if len(self.cardinalities) > 0:
            tokens.append(self.cat_emb(x_cat + self.cat_offsets))                 # (B, n_cat, d)
        return torch.cat(tokens, dim=1)  # (B, 1 + n_num + n_cat, d)


class FTTransformer(nn.Module):
    def __init__(self, n_num, cardinalities, d_model, n_layers, n_heads,
                 ffn_mult, dropout):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num, cardinalities, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=int(d_model * ffn_mult * 2),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,   # pre-norm: 안정적 학습
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Linear(d_model, 1),
        )

    def forward(self, x_num, x_cat):
        tokens = self.tokenizer(x_num, x_cat)
        out = self.encoder(tokens)
        return self.head(out[:, 0]).squeeze(-1)  # [CLS] 토큰 → logit


model = FTTransformer(
    n_num=len(num_cols),
    cardinalities=cardinalities,
    d_model=CFG["d_model"],
    n_layers=CFG["n_layers"],
    n_heads=CFG["n_heads"],
    ffn_mult=CFG["ffn_mult"],
    dropout=CFG["dropout"],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"파라미터 수: {n_params:,}")

In [ ]:
# ============================================================
# 6. 학습 — AdamW + warmup/cosine, AMP, early stopping(valid-tune PR-AUC)
# ============================================================
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_value, device=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

total_steps  = CFG["epochs"] * len(train_loader)
warmup_steps = max(1, int(total_steps * CFG["warmup_ratio"]))

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * p))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
try:
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

@torch.no_grad()
def predict_proba(loader):
    model.eval()
    probs, ys = [], []
    for x_num, x_cat, y in loader:
        x_num = x_num.to(device, non_blocking=True)
        x_cat = x_cat.to(device, non_blocking=True).long()
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logit = model(x_num, x_cat)
        probs.append(torch.sigmoid(logit.float()).cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(probs), np.concatenate(ys)

# ============================================================
# 체크포인트 — 런타임이 끊겨도 이 셀만 다시 실행하면 이어서 학습됩니다.
# (Drive 에 저장되므로 세션이 새로 시작돼도 유지됩니다)
# ============================================================
import shutil
os.makedirs(OUT_DIR, exist_ok=True)
CKPT_LOCAL = "/content/train_checkpoint.pt"                       # 매 epoch (빠름)
CKPT_DRIVE = os.path.join(OUT_DIR, "train_checkpoint.pt")         # 가끔 (세션 종료 대비)

print("early stopping 기준: valid-tune PR-AUC")
best_prauc, best_state, patience_cnt, start_epoch = -1.0, None, 0, 1

# --- 피처 지문 ---
# 피처 구성이 바뀌면(예: DELNG_MONTH/QUARTER 제외) 예전 체크포인트는 모양이 달라
# 이어받을 수 없다. 조용히 실패하지 않도록 지문을 비교해 자동으로 무시한다.
import hashlib, json as _json
FEAT_FP = hashlib.md5(_json.dumps(
    {"num": num_cols, "cat": cat_cols, "card": cardinalities},
    ensure_ascii=False, sort_keys=True).encode()).hexdigest()[:12]
print(f"피처 지문: {FEAT_FP}  (수치형 {len(num_cols)} · 범주형 {len(cat_cols)})")

# 로컬이 최신이므로 먼저 보고, 없으면 Drive 에서 복구
_resume_from = next((p for p in (CKPT_LOCAL, CKPT_DRIVE)
                     if CFG["resume"] and os.path.exists(p)), None)
if _resume_from:
    _ck_probe = torch.load(_resume_from, map_location="cpu", weights_only=False)
    if _ck_probe.get("feat_fp") != FEAT_FP:
        print(f"⚠️ 체크포인트의 피처 구성이 다릅니다 "
              f"({_ck_probe.get('feat_fp')} ≠ {FEAT_FP}) — 무시하고 처음부터 학습합니다.")
        _resume_from = None
    del _ck_probe

if _resume_from:
    ck = torch.load(_resume_from, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    scaler.load_state_dict(ck["scaler"])
    best_prauc   = ck["best_prauc"]
    best_state   = ck["best_state"]
    patience_cnt = ck["patience_cnt"]
    start_epoch  = ck["epoch"] + 1
    print(f"↻ 체크포인트에서 재개 ({os.path.basename(os.path.dirname(_resume_from)) or 'content'}): "
          f"epoch {ck['epoch']} 까지 완료, best PR-AUC {best_prauc:.4f} → epoch {start_epoch} 부터 계속")
    if start_epoch > CFG["epochs"]:
        print("   이미 지정한 epoch 수를 모두 마쳤습니다. 더 돌리려면 CFG['epochs'] 를 늘리세요.")
else:
    print(f"체크포인트 없음 — 처음부터 학습")
    print(f"   매 epoch: {CKPT_LOCAL}  /  {CFG['ckpt_drive_every']} epoch 마다 Drive 복사")

def _gpu_mem():
    if device.type != "cuda":
        return ""
    return (f"  GPU {torch.cuda.memory_allocated()/1024**3:.1f}/"
            f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.0f}GB")

for epoch in range(start_epoch, CFG["epochs"] + 1):
    model.train()
    total_loss, n_seen = 0.0, 0
    for x_num, x_cat, y in train_loader:
        x_num = x_num.to(device, non_blocking=True)
        x_cat = x_cat.to(device, non_blocking=True).long()
        y     = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            loss = criterion(model(x_num, x_cat), y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item() * len(y)
        n_seen += len(y)

    vt_prob, vt_y = predict_proba(vt_loader)
    prauc  = average_precision_score(vt_y, vt_prob)
    rocauc = roc_auc_score(vt_y, vt_prob)

    flag = ""
    if prauc > best_prauc:
        best_prauc = prauc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
        flag = "  ← best"
    else:
        patience_cnt += 1

    print(f"[{epoch:02d}/{CFG['epochs']}] loss={total_loss/n_seen:.4f}  "
          f"valid-tune PR-AUC={prauc:.4f}  ROC-AUC={rocauc:.4f}{_gpu_mem()}{flag}")

    # --- 체크포인트: 로컬은 매 epoch, Drive 는 가끔 ---
    torch.save({
        "epoch": epoch,
        "feat_fp": FEAT_FP,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "best_prauc": best_prauc,
        "best_state": best_state,
        "patience_cnt": patience_cnt,
    }, CKPT_LOCAL)

    stop_now = patience_cnt >= CFG["patience"]
    every = CFG["ckpt_drive_every"]
    if stop_now or epoch == CFG["epochs"] or (every and epoch % every == 0):
        try:
            shutil.copy(CKPT_LOCAL, CKPT_DRIVE)
            print(f"        ↳ Drive 백업 완료 (epoch {epoch})")
        except Exception as e:
            # Drive 쓰기가 실패해도 학습은 계속한다. 로컬 체크포인트는 남아 있다.
            print(f"        ↳ ⚠️ Drive 백업 실패({type(e).__name__}) — 로컬에는 저장됨")

    if stop_now:
        print(f"early stopping (patience {CFG['patience']})")
        break

if best_state is None:
    raise RuntimeError("학습이 한 epoch 도 완료되지 않았습니다. "
                       "batch_size 를 줄이거나 런타임을 재시작해 보세요.")
model.load_state_dict(best_state)
print(f"\nbest valid-tune PR-AUC = {best_prauc:.4f}")

In [ ]:
# ============================================================
# 7. 평가 + 위험점수 산정
#
#   위험점수 = 부정사용 확률 (0 ~ 1). 1 에 가까울수록 부정수급 가능성이 높다.
#   등급 구분은 사용하지 않는다.
#
#   원칙: threshold 는 valid-tune 에서 산출하고, 성능은 valid-report 에서만 보고한다.
# ============================================================
from sklearn.model_selection import StratifiedKFold

def best_f1_threshold(y_true, prob):
    prec, rec, thr = precision_recall_curve(y_true, prob)
    f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    i = int(np.argmax(f1s))
    return float(thr[i]), float(f1s[i])

# ---------- 위험점수 ----------
def to_risk_score(prob):
    """부정사용 확률을 그대로 위험점수로 쓴다 (0~1, 1에 가까울수록 위험)."""
    return np.round(np.asarray(prob, dtype=float), 6)

# 등급은 사용하지 않는다. 아래 두 개는 하위 셀 호환을 위한 자리만 유지.
GRADE_LABELS = {}
grade_cuts = []
def to_grade(prob):
    return np.zeros(len(np.asarray(prob)), dtype=int)

# ---------- 예측 ----------
vt_prob, vt_y = predict_proba(vt_loader)
vr_prob, vr_y = predict_proba(vr_loader)

print("===== threshold 무관 지표 (valid-report) =====")
print(f"PR-AUC  = {average_precision_score(vr_y, vr_prob):.4f}   "
      f"(무작위 기준선 {vr_y.mean():.4f})")
print(f"ROC-AUC = {roc_auc_score(vr_y, vr_prob):.4f}")

# ---------- 판정 threshold ----------
# report 셋 내부 교차적합: 각 fold 예측에는 그 fold 를 보지 않고 정한 threshold 만 적용
vr_pred = np.zeros(len(vr_y), dtype=int)
fold_thr = []
for a, b in StratifiedKFold(5, shuffle=True, random_state=CFG["seed"]).split(vr_prob.reshape(-1, 1), vr_y):
    t, _ = best_f1_threshold(vr_y[a], vr_prob[a])
    vr_pred[b] = (vr_prob[b] >= t).astype(int)
    fold_thr.append(t)
f1_main = f1_score(vr_y, vr_pred)

# 실제 배포(테스트)에 적용할 threshold 는 valid-tune 에서 산출 — report 정보 미사용
best_thr, f1_tune = best_f1_threshold(vt_y, vt_prob)

print("\n===== 최종 성능 (valid-report) =====")
print(f"F1={f1_main:.4f}  Precision={precision_score(vr_y, vr_pred, zero_division=0):.4f}  "
      f"Recall={recall_score(vr_y, vr_pred):.4f}")
print("Confusion [[TN FP][FN TP]]:", confusion_matrix(vr_y, vr_pred).tolist())
print(f"fold별 threshold {[round(t,4) for t in fold_thr]} → 대표값 {np.median(fold_thr):.4f}")
print(f"배포용 threshold (valid-tune 산출) = {best_thr:.4f}")

base = float(vr_y.mean())
f1_all_pos = 2 * base / (base + 1.0)
print(f"\n기준선: 전부 양성 예측 시 F1 = {f1_all_pos:.4f}  →  현재 모델 {f1_main/f1_all_pos:.2f}배")

# ---------- 위험점수 분포 ----------
vr_score = to_risk_score(vr_prob)
qs = np.percentile(vr_score, [10, 25, 50, 75, 90, 99])
print("\n===== 위험점수 분포 (valid-report) =====")
print(f"  평균 {vr_score.mean():.4f}   최소 {vr_score.min():.4f}   최대 {vr_score.max():.4f}")
print(f"  p10 {qs[0]:.4f} | p25 {qs[1]:.4f} | p50 {qs[2]:.4f} | "
      f"p75 {qs[3]:.4f} | p90 {qs[4]:.4f} | p99 {qs[5]:.4f}")
print(f"  실제 부정사용 평균점수 {vr_score[vr_y == 1].mean():.4f}  vs  "
      f"정상 평균점수 {vr_score[vr_y == 0].mean():.4f}")

# ---------- 상위 구간별 실측 (등급 없이 백분위로만) ----------
# "점수 상위 N% 를 검토하면 그중 몇 %가 실제 부정인가"를 보여준다.
print("\n===== 점수 상위 구간별 실측 =====")
print(f"  {'상위':>6s} {'건수':>8s} {'점수 하한':>10s} {'실제부정':>8s} {'정밀도':>8s} "
      f"{'lift':>6s} {'재현율':>8s}")
print("  " + "-" * 62)
order = np.argsort(-vr_score)
total_pos = int(vr_y.sum())
for q in [0.01, 0.05, 0.10, 0.20, 0.50]:
    k = max(1, int(len(vr_score) * q))
    sel = order[:k]
    tp = int(vr_y[sel].sum())
    print(f"  {q:6.0%} {k:8,} {vr_score[sel][-1]:10.4f} {tp:8,} "
          f"{tp/k:8.1%} {tp/k/base:6.1f} {tp/total_pos:8.1%}")
print("\n  · 정밀도 = 그 구간에서 실제 부정사용인 비율")
print("  · lift  = 무작위 검토 대비 몇 배 효율인지")
print("  · 재현율 = 그 구간까지 검토했을 때 전체 부정사용 중 잡아내는 비율")

# ---------- 구간별 성능: 전체 / 신규 이용자 / 기존 이용자 (팀 공지 기준) ----------
#   신규 = valid 의 TRGTER_ID 가 train 의 TRGTER_ID 집합에 없는 행.
#   같은 threshold(교차적합 결과)를 그대로 적용해 구간만 나눠 본다.
is_new_vr = is_new_user[idx_vr]

def _seg_row(name, m):
    n = int(m.sum())
    if n == 0:
        return f"  {name:14s} {'—':>8s}  (해당 행 없음)"
    y_, p_, pr_ = vr_y[m], vr_pred[m], vr_prob[m]
    npos = int(y_.sum())
    if npos == 0 or npos == n:
        return (f"  {name:14s} {n:8,} {npos:8,} {'—':>8s} {'—':>10s} {'—':>10s} "
                f"{'—':>9s}   (한쪽 클래스만 존재)")
    return (f"  {name:14s} {n:8,} {npos:8,} {y_.mean():8.2%} "
            f"{f1_score(y_, p_):10.4f} {average_precision_score(y_, pr_):10.4f} "
            f"{roc_auc_score(y_, pr_):9.4f}")

print("\n===== 구간별 성능 (valid-report, 동일 threshold) =====")
print(f"  {'구간':14s} {'행수':>8s} {'양성':>8s} {'양성률':>8s} "
      f"{'F1':>10s} {'PR-AUC':>10s} {'ROC-AUC':>9s}")
print("  " + "-" * 72)
print(_seg_row("전체", np.ones(len(vr_y), dtype=bool)))
print(_seg_row("신규 이용자", is_new_vr))
print(_seg_row("기존 이용자", ~is_new_vr))
print("\n  · 신규 = train 의 TRGTER_ID 집합에 없는 이용자 (모델이 처음 보는 사람)")
print("  · 두 구간의 격차가 크면 사람 단위 암기에 의존하고 있다는 뜻입니다.")

print("\n" + classification_report(vr_y, vr_pred, digits=4))

In [ ]:
# ============================================================
# 8. 테스트 추론 — 청크 단위 처리 (파일이 아무리 커도 동작)
#    parquet 을 배치로 나눠 읽으므로 메모리에 전체를 올리지 않는다.
# ============================================================
import pyarrow.parquet as pq

@torch.no_grad()
def score_frame(d: pd.DataFrame) -> np.ndarray:
    """DataFrame 한 덩어리 → 부정사용 확률"""
    xn = qt.transform(prep_num_raw(d)).astype(np.float32)
    xc = encode_cat(d)
    ds = TensorDataset(torch.from_numpy(xn), torch.from_numpy(xc))
    probs = []
    for x_num, x_cat in DataLoader(ds, batch_size=CFG["batch_size"] * 2, shuffle=False):
        x_num, x_cat = x_num.to(device), x_cat.to(device).long()
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logit = model(x_num, x_cat)
        probs.append(torch.sigmoid(logit.float()).cpu().numpy())
    return np.concatenate(probs)

def score_parquet(path: str, chunk_rows: int = None) -> pd.DataFrame:
    """대용량 parquet 을 청크로 읽어 위험점수를 산정한다."""
    chunk_rows = chunk_rows or CFG["infer_chunk_rows"]
    pf = pq.ParquetFile(path)
    total = pf.metadata.num_rows
    avail = set(pf.schema_arrow.names)

    need = list(dict.fromkeys(num_cols + cat_cols))           # 모델이 요구하는 피처
    lack = [c for c in need if c not in avail]
    if lack:
        raise ValueError(f"테스트 파일에 없는 피처 {len(lack)}개: {lack[:10]}")
    keep_ids = [c for c in ID_COLS if c in avail]
    has_label = TARGET in avail
    cols = need + keep_ids + ([TARGET] if has_label else [])

    print(f"테스트 추론 시작: {os.path.basename(path)}  {total:,}행  "
          f"(청크 {chunk_rows:,}행)")
    parts, done = [], 0
    for batch in pf.iter_batches(batch_size=chunk_rows, columns=cols):
        d = batch.to_pandas()
        prob = score_frame(d)
        out = d[keep_ids].copy()
        out["prob"] = prob
        out["risk_score"] = to_risk_score(prob)      # 0~1, 1에 가까울수록 위험
        out["pred"] = (prob >= best_thr).astype(int)
        if has_label:
            out["y_true"] = d[TARGET].astype(int).values
        parts.append(out)
        done += len(d)
        print(f"   {done:,}/{total:,}  ({done/total:.0%})")
    res = pd.concat(parts, ignore_index=True)
    print(f"완료: {len(res):,}행")
    return res

# ---------- 실행 ----------
if os.path.exists(TEST_PATH):
    test_result = score_parquet(TEST_PATH)

    ts = test_result["risk_score"].values
    print(f"\n===== 위험점수 분포 (test) =====")
    print(f"  평균 {ts.mean():.4f}   최소 {ts.min():.4f}   최대 {ts.max():.4f}")
    _q = np.percentile(ts, [10, 25, 50, 75, 90, 99])
    print(f"  p10 {_q[0]:.4f} | p25 {_q[1]:.4f} | p50 {_q[2]:.4f} | "
          f"p75 {_q[3]:.4f} | p90 {_q[4]:.4f} | p99 {_q[5]:.4f}")
    print(f"  양성 판정(threshold {best_thr:.4f}) {int(test_result['pred'].sum()):,}건 "
          f"({test_result['pred'].mean():.2%})")

    # ----- 상위 구간별 실측 (정답 컬럼이 있을 때만) -----
    if "y_true" in test_result:
        ty = test_result["y_true"].values
        tbase, tpos = float(ty.mean()), int(ty.sum())
        torder = np.argsort(-ts)
        print("\n===== 점수 상위 구간별 실측 (test) =====")
        print(f"  {'상위':>6s} {'건수':>8s} {'점수 하한':>10s} {'실제부정':>8s} "
              f"{'정밀도':>8s} {'lift':>6s} {'재현율':>8s}")
        print("  " + "-" * 62)
        for q in [0.01, 0.05, 0.10, 0.20, 0.50]:
            k = max(1, int(len(ts) * q))
            sel = torder[:k]
            tp = int(ty[sel].sum())
            print(f"  {q:6.0%} {k:8,} {ts[sel][-1]:10.4f} {tp:8,} "
                  f"{tp/k:8.1%} {tp/k/tbase:6.1f} {tp/tpos:8.1%}")

    if "y_true" in test_result:
        ty, tp_ = test_result["y_true"].values, test_result["prob"].values
        print("\n===== 테스트 성능 (정답 컬럼이 있어 계산) =====")
        print(f"F1={f1_score(ty, test_result['pred']):.4f}  "
              f"PR-AUC={average_precision_score(ty, tp_):.4f}  "
              f"ROC-AUC={roc_auc_score(ty, tp_):.4f}  "
              f"(무작위 기준선 {ty.mean():.4f})")
else:
    test_result = None
    print(f"⚠️ 테스트 파일이 없어 건너뜁니다: {TEST_PATH}")
    print("   나중에 파일이 준비되면 아래 한 줄로 실행할 수 있습니다:")
    print("     test_result = score_parquet('/content/drive/MyDrive/fds/test.parquet')")

In [ ]:
# ============================================================
# 9. 저장 — 예측 결과 · 모델 · 전처리 객체
# ============================================================
import pickle, sklearn
os.makedirs(OUT_DIR, exist_ok=True)

# --- 9-1. valid 전체 위험점수 ---
valid_prob_all, _ = predict_proba(
    make_loader(X_num_valid, X_cat_valid, y_valid, CFG["batch_size"] * 2, shuffle=False))
valid_out = valid_df[[c for c in ID_COLS if c in valid_df.columns]].copy()
valid_out["y_true"] = y_valid.astype(int)
valid_out["prob"] = valid_prob_all
valid_out["risk_score"] = to_risk_score(valid_prob_all)   # 0~1
valid_out["pred"] = (valid_prob_all >= best_thr).astype(int)
valid_out.to_csv(os.path.join(OUT_DIR, "valid_risk.csv"), index=False)

saved = ["valid_risk.csv"]

# --- 9-2. 테스트 위험점수 ---
if test_result is not None:
    test_result.to_csv(os.path.join(OUT_DIR, "test_risk.csv"), index=False)
    saved.append("test_risk.csv")

# --- 9-3. 모델 + 전처리 (반드시 한 쌍으로 보관) ---
torch.save({
    "model_state_dict": model.state_dict(), "cfg": CFG,
    "num_cols": num_cols, "cat_cols": cat_cols, "cardinalities": cardinalities,
    "best_threshold": best_thr,
    "prior_train": prior_train, "prior_target": prior_target,
    "target": TARGET, "id_cols": ID_COLS, "torch_version": torch.__version__,
    "excluded_cols": EXCLUDE_COLS, "feat_fp": FEAT_FP,
    "n_train": int(len(y_train)), "prior_valid": prior_target,
}, os.path.join(OUT_DIR, "ft_transformer.pt"))
with open(os.path.join(OUT_DIR, "preprocess.pkl"), "wb") as f:
    pickle.dump({"cat_maps": cat_maps, "num_medians": num_medians, "qt": qt,
                 "num_cols": num_cols, "cat_cols": cat_cols,
                 "best_threshold": best_thr,
                 "sklearn_version": sklearn.__version__}, f)
saved += ["ft_transformer.pt", "preprocess.pkl"]

print(f"저장 위치: {OUT_DIR}")
for fn in saved:
    print(f"  {fn:26s} {os.path.getsize(os.path.join(OUT_DIR, fn))/1024**2:8.2f} MB")

if not OUT_DIR.startswith("/content/drive"):
    print("\n⚠️ Drive 밖에 저장했습니다. 세션 종료 시 삭제되니 지금 내려받으세요.")

print("\n결과 CSV 컬럼: row_id · prob · risk_score(0~1, 1에 가까울수록 위험) · pred")
print("앙상블에는 risk_score 가 아니라 prob(원본 확률)을 사용하세요.")

In [ ]:
# ============================================================
# 10. 팀 제출용 점수 CSV — dl_scores_valid.csv / dl_scores_test.csv
#     규격: row_id, y, score   (score = 0~1, 1 에 가까울수록 부정수급)
#     row_id 는 분할 parquet 의 맨 앞 컬럼을 그대로 사용한다.
# ============================================================
ID_KEY = "row_id"

def _export(df_ids, y, prob, fname):
    if ID_KEY not in df_ids.columns:
        raise KeyError(f"{ID_KEY} 컬럼이 없습니다. 분할 parquet 의 맨 앞 컬럼명을 "
                       f"ID_KEY 에 지정하세요. 보유 컬럼: {list(df_ids.columns)[:10]}")
    out = pd.DataFrame({
        ID_KEY:  df_ids[ID_KEY].values,
        "y":     (np.full(len(prob), np.nan) if y is None
                  else np.asarray(y).astype(np.int8)),
        "score": np.round(np.asarray(prob, dtype=float), 6),
    })
    if out[ID_KEY].duplicated().any():
        print(f"⚠️ {fname}: row_id 중복 {int(out[ID_KEY].duplicated().sum()):,}건")
    path = os.path.join(OUT_DIR, fname)
    out.to_csv(path, index=False)
    lab = "라벨 없음" if y is None else f"양성 {int(out['y'].sum()):,}건 ({out['y'].mean():.2%})"
    print(f"  {fname:24s} {len(out):8,}행   {lab}   "
          f"score {out['score'].min():.4f}~{out['score'].max():.4f}")
    return out

print(f"저장 위치: {OUT_DIR}\n")

# --- valid: 분할 전 전체 행 ---
dl_valid = _export(valid_df, y_valid.astype(int), valid_prob_all, "dl_scores_valid.csv")

# --- test ---
if test_result is not None:
    dl_test = _export(
        test_result,
        test_result["y_true"].values if "y_true" in test_result else None,
        test_result["prob"].values,
        "dl_scores_test.csv",
    )
else:
    dl_test = None
    print("  dl_scores_test.csv       건너뜀 — 8번 칸(테스트 추론)을 먼저 실행하세요")

# --- 규격 확인 ---
print("\n[규격 확인] 컬럼 =", list(dl_valid.columns))
print(dl_valid.head(3).to_string(index=False))
print("\n※ score 는 확률 그대로입니다. train 양성률과 valid 양성률이 다르면 "
      "절대값이 팀원 모델과 어긋납니다.")
print("   앙상블은 값 평균이 아니라 순위평균(rank average)으로 하세요.")

In [ ]:
%%writefile /content/dl_predict.py
"""
dl_predict.py — FDS 딥러닝 파트 (FT-Transformer) 추론 모듈

사용법
------
    from dl_predict import predict_dl
    import pandas as pd

    df    = pd.read_parquet("valid.parquet")   # 분할 parquet 로우데이터 그대로
    score = predict_dl(df)                     # np.ndarray, 0~1 (1에 가까울수록 부정수급)

같은 폴더에 아래 두 파일이 있어야 합니다.
    ft_transformer.pt   모델 가중치 · 피처 목록 · threshold
    preprocess.pkl      cat_maps · 중앙값 · QuantileTransformer

다른 위치에 있다면:
    from dl_predict import load_dl
    load_dl("/content/drive/MyDrive/fds/outputs")

CLI:
    python dl_predict.py input.parquet output.csv [artifact_dir]
"""
from __future__ import annotations

import os
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

__all__ = ["predict_dl", "load_dl", "predict_parquet", "THRESHOLD", "FEATURE_INFO"]

BATCH_SIZE = 4096
_ART: dict | None = None          # 로드된 아티팩트 캐시
THRESHOLD: float | None = None    # 배포용 threshold (valid-tune 에서 산출)
FEATURE_INFO: dict | None = None  # {"num_cols": [...], "cat_cols": [...]}


# ============================================================
# 모델 정의 — 학습 노트북과 반드시 동일해야 state_dict 가 맞는다
# ============================================================
class FeatureTokenizer(nn.Module):
    def __init__(self, n_num: int, cardinalities: list, d_model: int):
        super().__init__()
        self.n_num = n_num
        self.num_weight = nn.Parameter(torch.empty(n_num, d_model))
        self.num_bias = nn.Parameter(torch.empty(n_num, d_model))
        nn.init.normal_(self.num_weight, std=0.02)
        nn.init.normal_(self.num_bias, std=0.02)
        self.cardinalities = cardinalities
        if len(cardinalities) > 0:
            offsets = np.cumsum([0] + list(cardinalities[:-1]))
            self.register_buffer("cat_offsets", torch.tensor(offsets, dtype=torch.long))
            self.cat_emb = nn.Embedding(int(sum(cardinalities)), d_model)
            nn.init.normal_(self.cat_emb.weight, std=0.02)
        self.cls = nn.Parameter(torch.empty(1, 1, d_model))
        nn.init.normal_(self.cls, std=0.02)

    def forward(self, x_num, x_cat):
        B = x_num.size(0)
        tokens = [self.cls.expand(B, -1, -1)]
        if self.n_num > 0:
            tokens.append(x_num.unsqueeze(-1) * self.num_weight + self.num_bias)
        if len(self.cardinalities) > 0:
            tokens.append(self.cat_emb(x_cat + self.cat_offsets))
        return torch.cat(tokens, dim=1)


class FTTransformer(nn.Module):
    def __init__(self, n_num, cardinalities, d_model, n_layers, n_heads,
                 ffn_mult, dropout):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num, cardinalities, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=int(d_model * ffn_mult * 2),
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.ReLU(), nn.Linear(d_model, 1))

    def forward(self, x_num, x_cat):
        return self.head(self.encoder(self.tokenizer(x_num, x_cat))[:, 0]).squeeze(-1)


# ============================================================
# 전처리 — 학습 노트북과 반드시 동일 (문자열 정규화 규칙 포함)
# ============================================================
def _norm_cat(s: pd.Series) -> pd.Series:
    return (s.astype(str).str.strip().str.lower()
             .replace({"nan": "__na__", "none": "__na__"}))


def _encode_cat(d: pd.DataFrame, cat_cols, cat_maps) -> np.ndarray:
    out = np.zeros((len(d), len(cat_cols)), dtype=np.int64)
    for j, c in enumerate(cat_cols):
        out[:, j] = _norm_cat(d[c]).map(cat_maps[c]).fillna(0).astype(np.int64).values
    return out                       # 0 = 학습 때 못 본 값(UNK)


def _prep_num(d: pd.DataFrame, num_cols, num_medians) -> np.ndarray:
    x = d[num_cols].astype(np.float64)
    return x.replace([np.inf, -np.inf], np.nan).fillna(num_medians).values


# ============================================================
# 로드
# ============================================================
def load_dl(artifact_dir: str | None = None, device: str | None = None) -> dict:
    """ft_transformer.pt + preprocess.pkl 을 읽어 모델을 준비한다."""
    global _ART, THRESHOLD, FEATURE_INFO

    if artifact_dir is None:
        # 파일로 import 됐으면 그 폴더, 셀에 붙여넣어 실행 중이면 현재 폴더
        artifact_dir = (os.path.dirname(os.path.abspath(__file__))
                        if "__file__" in globals() else os.getcwd())
    pt_path = os.path.join(artifact_dir, "ft_transformer.pt")
    pkl_path = os.path.join(artifact_dir, "preprocess.pkl")
    for p in (pt_path, pkl_path):
        if not os.path.exists(p):
            raise FileNotFoundError(
                f"{os.path.basename(p)} 를 찾을 수 없습니다: {p}\n"
                f"ft_transformer.pt 와 preprocess.pkl 은 한 쌍으로 같은 폴더에 두세요."
            )

    dev = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    ck = torch.load(pt_path, map_location=dev, weights_only=False)
    with open(pkl_path, "rb") as f:
        pp = pickle.load(f)

    cfg = ck["cfg"]
    model = FTTransformer(
        n_num=len(ck["num_cols"]), cardinalities=ck["cardinalities"],
        d_model=cfg["d_model"], n_layers=cfg["n_layers"], n_heads=cfg["n_heads"],
        ffn_mult=cfg["ffn_mult"], dropout=cfg["dropout"],
    ).to(dev)
    model.load_state_dict(ck["model_state_dict"])
    model.eval()

    _ART = {
        "model": model, "device": dev,
        "num_cols": ck["num_cols"], "cat_cols": ck["cat_cols"],
        "cat_maps": pp["cat_maps"], "num_medians": pp["num_medians"], "qt": pp["qt"],
        "threshold": ck.get("best_threshold"),
    }
    THRESHOLD = _ART["threshold"]
    FEATURE_INFO = {"num_cols": ck["num_cols"], "cat_cols": ck["cat_cols"]}
    return _ART


def _art() -> dict:
    return _ART if _ART is not None else load_dl()


# ============================================================
# 추론
# ============================================================
@torch.no_grad()
def predict_dl(df: pd.DataFrame, batch_size: int = BATCH_SIZE) -> np.ndarray:
    """분할 parquet 로우데이터(DataFrame) → 부정수급 점수 (0~1, 1에 가까울수록 위험).

    입력에 라벨(INJSTUSE_AT)이나 식별자(row_id 등)가 섞여 있어도 무시하고
    학습 때 쓴 피처만 골라 씁니다. 행 순서는 그대로 보존됩니다.
    """
    a = _art()
    missing = [c for c in a["num_cols"] + a["cat_cols"] if c not in df.columns]
    if missing:
        raise ValueError(
            f"입력에 없는 피처 {len(missing)}개: {missing[:10]}"
            f"{' ...' if len(missing) > 10 else ''}\n"
            f"학습 때와 다른 전처리본을 넣었을 가능성이 큽니다."
        )
    if len(df) == 0:
        return np.zeros(0, dtype=np.float64)

    x_num = a["qt"].transform(
        _prep_num(df, a["num_cols"], a["num_medians"])).astype(np.float32)
    x_cat = _encode_cat(df, a["cat_cols"], a["cat_maps"])

    dev, model = a["device"], a["model"]
    out = np.empty(len(df), dtype=np.float64)
    for i in range(0, len(df), batch_size):
        j = min(i + batch_size, len(df))
        xn = torch.from_numpy(x_num[i:j]).to(dev)
        xc = torch.from_numpy(x_cat[i:j]).to(dev)
        with torch.autocast(device_type=dev.type, enabled=(dev.type == "cuda")):
            logit = model(xn, xc)
        out[i:j] = torch.sigmoid(logit.float()).cpu().numpy()
    return out


def predict_parquet(path: str, chunk_rows: int = 200_000,
                    id_key: str = "row_id", target: str = "INJSTUSE_AT") -> pd.DataFrame:
    """대용량 parquet 을 청크로 읽어 채점한다 → row_id, y, score DataFrame."""
    import pyarrow.parquet as pq

    a = _art()
    pf = pq.ParquetFile(path)
    avail = set(pf.schema_arrow.names)
    need = list(dict.fromkeys(a["num_cols"] + a["cat_cols"]))
    cols = need + ([id_key] if id_key in avail else []) + \
           ([target] if target in avail else [])

    parts, done, total = [], 0, pf.metadata.num_rows
    for batch in pf.iter_batches(batch_size=chunk_rows, columns=cols):
        d = batch.to_pandas()
        parts.append(pd.DataFrame({
            id_key: d[id_key].values if id_key in avail else np.arange(done, done + len(d)),
            "y": d[target].astype(np.int8).values if target in avail else np.nan,
            "score": np.round(predict_dl(d), 6),
        }))
        done += len(d)
        print(f"   {done:,}/{total:,} ({done/max(total,1):.0%})")
    return pd.concat(parts, ignore_index=True)


if __name__ == "__main__":
    import sys

    if len(sys.argv) < 3:
        print(__doc__)
        sys.exit(1)
    src, dst = sys.argv[1], sys.argv[2]
    load_dl(sys.argv[3] if len(sys.argv) > 3 else None)
    res = predict_parquet(src)
    res.to_csv(dst, index=False)
    print(f"저장 완료: {dst}  ({len(res):,}행)  threshold={THRESHOLD}")

In [ ]:
# ============================================================
# 11. 제출 파일 정합성 검증
#     dl_predict.py 를 실제로 import 해서, 노트북이 낸 점수와
#     완전히 같은 값이 나오는지 확인한다. (제출 전 필수)
# ============================================================
import sys, shutil

# 바로 위 칸이 /content/dl_predict.py 를 만들었으면 OUT_DIR 로 복사한다.
# (파일을 직접 업로드하셨다면 이 단계는 조용히 건너뜁니다)
_dl_path = os.path.join(OUT_DIR, "dl_predict.py")
if os.path.exists("/content/dl_predict.py"):
    try:
        shutil.copy("/content/dl_predict.py", _dl_path)
        print(f"dl_predict.py → {_dl_path} 복사 완료")
    except Exception as e:
        print(f"⚠️ Drive 복사 실패({type(e).__name__}) — /content 의 파일로 검증합니다")
        _dl_path = "/content/dl_predict.py"

if not os.path.exists(_dl_path):
    print(f"⚠️ dl_predict.py 가 없습니다: {_dl_path}")
    print("   바로 위 칸(10-b)을 실행하거나, 전달받은 파일을")
    print("   ft_transformer.pt · preprocess.pkl 과 같은 폴더에 올리세요.")
else:
    _dl_dir = os.path.dirname(_dl_path)
    if _dl_dir not in sys.path:
        sys.path.insert(0, _dl_dir)
    for _m in ("dl_predict",):
        if _m in sys.modules:
            del sys.modules[_m]
    import dl_predict
    dl_predict.load_dl(OUT_DIR)   # 아티팩트(.pt/.pkl)는 항상 OUT_DIR 에서

    # valid 앞 3,000행으로 대조 (전체를 돌릴 필요 없음)
    _n = min(3000, len(valid_df))
    _ref = valid_prob_all[:_n]
    _got = dl_predict.predict_dl(valid_df.iloc[:_n])
    _diff = np.abs(_ref - _got)

    print(f"대조 {_n:,}행 — 최대 오차 {_diff.max():.3e}  평균 {_diff.mean():.3e}")
    if _diff.max() < 1e-4:
        print("✅ 노트북과 dl_predict.py 의 점수가 일치합니다. 제출 가능합니다.")
    else:
        print("❌ 점수가 다릅니다. 전처리 규칙이나 모델 정의가 어긋났습니다.")
        print("   ft_transformer.pt / preprocess.pkl 이 이번 학습본인지 확인하세요.")

    print(f"\ndl_predict.THRESHOLD = {dl_predict.THRESHOLD}")
    print(f"피처 — 수치형 {len(dl_predict.FEATURE_INFO['num_cols'])}개, "
          f"범주형 {len(dl_predict.FEATURE_INFO['cat_cols'])}개")

# --- 제출 파일 체크리스트 ---
print("\n===== 제출 파일 =====")
for fn in ["ft_transformer.pt", "preprocess.pkl", "dl_predict.py",
           "dl_scores_valid.csv", "dl_scores_test.csv"]:
    p = os.path.join(OUT_DIR, fn)
    mark = "✅" if os.path.exists(p) else "❌"
    size = f"{os.path.getsize(p)/1024**2:8.2f} MB" if os.path.exists(p) else "       없음"
    print(f"  {mark} {fn:24s} {size}")

## 산출물

`OUT_DIR` 에 다음이 저장됩니다.

| 파일 | 내용 |
|---|---|
| `ft_transformer.pt` | 모델 가중치 · 피처 목록 · threshold |
| `preprocess.pkl` | cat_maps · 중앙값 · QuantileTransformer |
| `dl_scores_valid.csv` | `row_id, y, score` (제출 규격) |
| `dl_scores_test.csv` | `row_id, y, score` (제출 규격) |
| `valid_risk.csv` / `test_risk.csv` | 상세본 (prob·pred 포함, 내부 확인용) |

여기에 별도로 전달받은 **`dl_predict.py`** 를 같은 폴더에 올리면 제출 세트가 완성됩니다. 마지막 칸이 세 파일의 정합성(노트북 점수 == `predict_dl` 점수)을 검증합니다.

## 위험점수 해석

| 컬럼 | 의미 |
|---|---|
| `score` / `prob` | 부정수급 점수 0~1. **1 에 가까울수록 위험** |
| `pred` | threshold 적용 판정 (0/1) |

등급 구분은 사용하지 않습니다. 평가 셀이 **점수 상위 1/5/10/20/50% 구간별 정밀도·lift·재현율**을 출력하니, 검토 인력에 맞춰 상위 몇 %를 볼지만 정하면 됩니다.

## ⚠️ 앙상블 시 주의

train 양성률과 valid 양성률이 크게 다르면 점수의 **절대값**이 팀원 모델과 어긋납니다(순위는 멀쩡합니다). 앙상블은 값 평균이 아니라 **순위평균(rank average)** 으로 하세요.

```python
from scipy.stats import rankdata
ens = (rankdata(dl["score"]) + rankdata(ml["score"])) / (2 * len(dl))
```

## 런타임이 끊길 때

**학습 셀만 다시 실행하면 이어서 학습됩니다.**

```
↻ 체크포인트에서 재개 (content): epoch 12 까지 완료, best PR-AUC 0.6xxx → epoch 13 부터 계속
```

체크포인트는 두 곳에 저장됩니다.

| 위치 | 주기 | 이유 |
|---|---|---|
| `/content/train_checkpoint.pt` | **매 epoch** | 로컬이라 빠름. 세션이 살아 있으면 여기서 재개 |
| `OUT_DIR/train_checkpoint.pt` | **5 epoch 마다 + 종료 시** | 세션이 완전히 죽어도 Drive 에 남음 |

Colab 의 Drive 마운트는 잦은 쓰기에서 멈추는 일이 잦아 이렇게 나눴습니다. Drive 쓰기가 실패해도 학습은 계속되고 로컬 체크포인트는 남습니다.

재개 시 로컬을 먼저 보고, 없으면 Drive 에서 복구합니다. 처음부터 다시 하려면 두 파일을 지우거나 `CFG["resume"] = False` 로 두세요. Drive 백업 주기는 `CFG["ckpt_drive_every"]` 로 조절합니다(0 이면 종료 시에만).

**그래도 자주 끊긴다면** 순서대로 시도하세요.

1. `CFG["batch_size"] = 512` (또는 256) — GPU 메모리 부족일 때
2. `CFG["qt_subsample"] = 100_000` — 전처리 단계에서 끊길 때
3. 런타임 → 세션 다시 시작 후 재실행 (누적된 메모리 정리)
4. 학습 중 브라우저 탭을 열어두기 — 90분 방치하면 Colab 이 세션을 종료합니다

메모리 절약은 이미 적용되어 있습니다.

| 항목 | 효과 |
|---|---|
| 전처리 후 train DataFrame 해제 | 약 1.1 GB |
| 범주형 int64 → int32 | 약 56 MB |
| QuantileTransformer 표본 제한 | 약 185 MB |
| `num_workers=0` | 워커 프로세스 메모리 중복 제거 |

학습 로그에 GPU 사용량이 함께 표시되니, 여유가 얼마나 남았는지 확인하실 수 있습니다.

## 테스트 파일이 클 때

`score_parquet()` 이 **청크 단위로 읽어 처리**하므로 파일 크기에 제한이 없습니다. 메모리가 부족하면 Config 에서 줄이세요.

```python
CFG["infer_chunk_rows"] = 50_000    # 기본 200,000
```

노트북 실행이 끝난 뒤 다른 파일을 채점하려면 한 줄이면 됩니다.

```python
result = score_parquet("/content/drive/MyDrive/fds/test_full.parquet")
result.to_csv("/content/drive/MyDrive/fds/outputs/test_full_risk.csv", index=False)
```

정답 컬럼(`INJSTUSE_AT`)이 있으면 성능까지 함께 출력하고, 없으면 점수만 산정합니다.

## ⚠️ 누수 확인

로드 셀의 `[누수 제거]` 출력을 꼭 확인하세요.

- **제거 0행** — 이미 분리된 train 파일을 쓰고 있다는 뜻. 정상입니다.
- **제거 13만행 내외** — 5.5차 전체 파일을 쓰고 있어 valid·test 행이 걸러진 것. 정상 동작입니다.
- **제거 50% 이상** — train 파일이 잘못되었을 가능성이 큽니다.

## 팀 공통 프로토콜

- threshold·등급 컷오프는 **valid-tune 에서만** 산출, 성능은 **valid-report 에서만** 보고
- 분할 시드 `random_state=42` 로 통일하면 세 모델이 같은 행으로 비교됩니다
- valid 로 교차검증할 때는 **`StratifiedGroupKFold(groups=TRGTER_ID)`** 필수 — 무작위 분할은 같은 사람 암기로 F1 을 크게 부풀립니다
- 앙상블은 `prob` 컬럼끼리, 순위 평균(rank average) 권장